# DistilBERT Emotion Classifier — Kaggle Training

**Repo:** https://github.com/Priyrajsinh/distilbert-emotion-classifier-fastapi  
**Model:** `distilbert-base-uncased` → fine-tuned on GoEmotions 7-class macro-emotions  
**Expected runtime:** ~15–20 min on T4 GPU

Run all cells top-to-bottom. The trained model is saved to `/kaggle/working/sentiment_model.zip` — download it from the Output panel on the right.

## 1 — GPU check

In [ ]:
import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found. Enable it via Settings → Accelerator → GPU T4 x1')

## 2 — Clone repo and navigate

In [ ]:
!git clone https://github.com/Priyrajsinh/distilbert-emotion-classifier-fastapi.git
%cd distilbert-emotion-classifier-fastapi

## 3 — Install training dependencies

In [ ]:
# Install only the packages needed for training — torch/pandas/numpy/sklearn
# are already present on Kaggle. We pin transformers>=4.41 for eval_strategy support.
!pip install -q \
    'transformers>=4.41.0' \
    datasets \
    mlflow \
    pandera \
    accelerate \
    pyyaml

## 4 — Verify install

In [ ]:
import transformers, datasets, mlflow, pandera
print(f'transformers : {transformers.__version__}')
print(f'datasets     : {datasets.__version__}')
print(f'mlflow       : {mlflow.__version__}')
print(f'pandera      : {pandera.__version__}')

## 5 — Data pipeline (download GoEmotions → split → save CSVs)

In [ ]:
import sys
sys.path.insert(0, '.')

from src.data.dataset import load_goemotions
from src.data.preprocessing import stratified_split

df = load_goemotions()
train_df, val_df, test_df = stratified_split(df)
print(f'\nSplits ready — train: {len(train_df)}  val: {len(val_df)}  test: {len(test_df)}')

## 6 — Train

3 epochs, batch 32, weighted cross-entropy, MLflow tracking, best checkpoint saved.  
Watch for `eval_f1` rising each epoch in the progress table below.

In [ ]:
!python src/training/train.py --config config/config.yaml

## 7 — Verify model artefacts

In [ ]:
import os
model_dir = 'models/sentiment_model'
files = os.listdir(model_dir)
print(f'Files in {model_dir}:')
for f in sorted(files):
    size = os.path.getsize(os.path.join(model_dir, f))
    print(f'  {f:40s}  {size/1e6:6.1f} MB')

import json
with open('models/training_stats.json') as fh:
    print('\ntraining_stats.json:', json.load(fh))

## 8 — Quick inference smoke-test

In [ ]:
from src.models.model import SentimentClassifier

clf = SentimentClassifier()
clf.load('models/sentiment_model')

test_sentences = [
    'I just got promoted, this is the best day ever!',
    'I can not believe they cancelled the show, I am devastated.',
    'This is absolutely outrageous and unacceptable.',
    'I have no idea what just happened.',
]

labels   = clf.predict(test_sentences)
probas   = clf.predict_proba(test_sentences)

print(f'{"Text":<55}  {"Label":<10}  Confidence')
print('-' * 80)
for text, label, proba in zip(test_sentences, labels, probas):
    conf = proba[label]
    print(f'{text[:53]:<55}  {label:<10}  {conf:.1%}')

## 9 — Package model for download

Creates `sentiment_model.zip` in the Output panel.  
After downloading, unzip into `models/sentiment_model/` on your local machine.

In [ ]:
import shutil
zip_path = '/kaggle/working/sentiment_model'
shutil.make_archive(zip_path, 'zip', 'models/sentiment_model')
size_mb = os.path.getsize(zip_path + '.zip') / 1e6
print(f'sentiment_model.zip  →  {size_mb:.0f} MB')
print('Download it from the Output panel on the right →')